In [6]:
# ════════════════════════════════════════════════════════════════════════
# 0 ▸ Imports & constant paths
# ════════════════════════════════════════════════════════════════════════
from pathlib import Path
import pandas as pd, numpy as np, nibabel as nib, cv2, traceback

CSV_PATH   = Path(r"C:\Users\Ryan Krishna\Documents\tests\test_results.csv")
NIFTI_ROOT = Path(r"C:\Users\Ryan Krishna\Documents\tests")
OUT_DIR    = Path(r"C:\Users\Ryan Krishna\Documents\tests_mp4s")
OUT_DIR.mkdir(exist_ok=True)

# ════════════════════════════════════════════════════════════════════════
# 1 ▸ Load overscanning CSV
# ════════════════════════════════════════════════════════════════════════
df = pd.read_csv(CSV_PATH, sep=None, engine="python", encoding="utf-8-sig")
df.columns = df.columns.str.strip().str.replace("\ufeff", "", regex=False)
if "file_name" not in df.columns:
    raise KeyError("`file_name` column missing after cleanup!")

# ════════════════════════════════════════════════════════════════════════
# 2 ▸ Helper : build scrolling MP4 for one patient   (fps = 48)
#    Adds optional `y_stretch` to remove vertical squash without distorting labels
# ════════════════════════════════════════════════════════════════════════
def build_mp4(scan_id: str,
              pubic_z_mm: float,
              organ_z_mm: float,
              top_organ_label: str,
              fps: int = 48,
              y_stretch: float = 5):   # >1 = taller, 1 = no stretch

    # ── locate required files ───────────────────────────────────────────
    folder      = NIFTI_ROOT / scan_id
    ct_path     = next((p for p in folder.glob("*.nii*")
                        if p.stem.startswith(scan_id)), None)
    femur_path  = folder / "femur_combined.nii.gz"
    organ_path  = folder / "liver_spleen_combined.nii.gz"
    mp4_path    = OUT_DIR / f"{scan_id}.mp4"

    if not ct_path or not ct_path.exists():
        raise FileNotFoundError("CT volume not found")
    for req in (femur_path, organ_path):
        if not req.exists():
            raise FileNotFoundError(f"Mask missing: {req.name}")

    # ── load volumes ─────────────────────────────────────────────────────
    ct_img   = nib.load(str(ct_path));  data_ct  = ct_img.get_fdata()
    fem_mask = nib.load(str(femur_path)).get_fdata() > 0
    org_mask = nib.load(str(organ_path)).get_fdata() > 0
    affine   = ct_img.affine
    X, Y, Z  = data_ct.shape

    # ── world-z → row mapping (fixed) ────────────────────────────────────
    z_world  = np.flip((affine @ np.stack(
        [np.zeros(Z), np.zeros(Z), np.arange(Z), np.ones(Z)], axis=1
    ).T)[2])
    pubic_row  = int(np.argmin(np.abs(z_world - pubic_z_mm)))
    organ_row  = int(np.argmin(np.abs(z_world - organ_z_mm)))

    # ── restrict y-range to liver/spleen coverage (±1 slice) ─────────────
    org_presence = org_mask.any(axis=(0, 2))            # shape (Y,)
    if org_presence.any():
        first_org = int(np.where(org_presence)[0][0])
        last_org  = int(np.where(org_presence)[0][-1])
        start_y   = max(0, first_org - 1)
        end_y     = min(Y - 1, last_org + 1)
    else:                                               # mask empty → full range
        start_y, end_y = 0, Y - 1

    # pick a representative slice near centre of this trimmed range
    chosen_y = (start_y + end_y) // 2

    # ── inner renderer for ONE coronal slice ────────────────────────────
    font, fs, th = cv2.FONT_HERSHEY_SIMPLEX, 0.55, 2
    red, green   = (0,0,255), (0,255,0)

    def render(y_idx: int):
        # 2D coronal views (flip UD so head at top)
        ct  = np.flipud(data_ct[:, y_idx, :].T)
        fms = np.flipud(fem_mask[:, y_idx, :].T)
        oms = np.flipud(org_mask[:, y_idx, :].T)

        # grayscale base
        img = ct - np.percentile(ct, 0.5)
        img = img / (np.percentile(img, 99.5) - 1e-8)
        img = np.clip(img, 0, 1); img = (img * 255).astype(np.uint8)
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)

        # overlay masks
        overlay = np.zeros_like(img)
        overlay[fms] = (255, 0, 0)       # blue
        overlay[oms] = (0, 255, 255)     # yellow
        img = cv2.addWeighted(img, 0.8, overlay, 0.2, 0)

        # ── stretch CT + masks (labels drawn *after* resize) ────────────
        H0, W0 = img.shape[:2]
        if y_stretch != 1.0:
            new_h = int(H0 * y_stretch)
            img   = cv2.resize(img, (W0, new_h), interpolation=cv2.INTER_LINEAR)

        # rescaled landmark rows
        pubic_row_s = int(pubic_row * y_stretch)
        organ_row_s = int(organ_row * y_stretch)

        # landmark lines
        cv2.line(img, (0, pubic_row_s), (W0 - 1, pubic_row_s), red,   2)
        cv2.line(img, (0, organ_row_s), (W0 - 1, organ_row_s), green, 2)

        # labels (text stays crisp)
        H1 = img.shape[0]
        cv2.putText(img, f"Pubic symphysis (z={pubic_z_mm:.0f} mm)",
                    (10, max(20, pubic_row_s - 6)), font, fs, red,   th, cv2.LINE_AA)
        cv2.putText(img, f"{top_organ_label} (z={organ_z_mm:.0f} mm)",
                    (10, min(H1 - 10, organ_row_s + 20)), font, fs, green, th, cv2.LINE_AA)

        # slice indicator bottom-left
        cv2.putText(img, f"{scan_id} | coronal y={y_idx}",
                    (10, H1 - 10), font, fs, (255,255,0), th, cv2.LINE_AA)
        return img

    # ── initialise VideoWriter ───────────────────────────────────────────
    first = render(start_y)
    H_, W_ = first.shape[:2]           # note: shape order is (H, W)
    vw = cv2.VideoWriter(str(mp4_path),
                         cv2.VideoWriter_fourcc(*"mp4v"),
                         fps, (W_, H_))

    for y in range(start_y, end_y + 1):
        vw.write(render(y))
    vw.release()

# ════════════════════════════════════════════════════════════════════════
# 3 ▸ Build MP4 for every patient listed in the CSV
# ════════════════════════════════════════════════════════════════════════
ok = failed = 0
for _, r in df.iterrows():
    sid = r["file_name"].split(".nii")[0]
    try:
        build_mp4(
            sid,
            float(r["pubic_z_mm"]),
            float(r["liver_spleen_z_mm"]),
            str(r["top_organ"]).strip()
        )
        ok += 1
    except Exception as e:
        failed += 1
        print(f"✗ {sid}: {e}")
        traceback.print_exc()

print(f"\n✓ {ok} MP4s created   ✗ {failed} failed")
print(f"Videos saved in: {OUT_DIR}")


✓ 1 MP4s created   ✗ 0 failed
Videos saved in: C:\Users\Ryan Krishna\Documents\tests_mp4s
